In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

In [2]:
df = pd.read_csv("../data/german_credit_data.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

df.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,49,male,1,own,little,NaN,2096,12,education,good
3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,53,male,2,free,little,little,4870,24,car,bad


In [3]:
X = df.drop("Risk", axis=1)
y = df["Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
numeric_features = [
    "Age",
    "Credit amount",
    "Duration"
]

categorical_features = [
    "Sex",
    "Job",                
    "Housing",
    "Saving accounts",
    "Checking account",
    "Purpose"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [5]:
knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

knn_pipeline.fit(X_train, y_train)

y_pred_knn = knn_pipeline.predict(X_test)
y_prob_knn = knn_pipeline.predict_proba(X_test)[:, 1]

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_knn))
print(confusion_matrix(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

KNN Accuracy: 0.67
ROC-AUC: 0.6316666666666667
[[ 16  44]
 [ 22 118]]
              precision    recall  f1-score   support

         bad       0.42      0.27      0.33        60
        good       0.73      0.84      0.78       140

    accuracy                           0.67       200
   macro avg       0.57      0.55      0.55       200
weighted avg       0.64      0.67      0.64       200



In [6]:
nb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GaussianNB())
])

nb_pipeline.fit(X_train, y_train)

y_pred_nb = nb_pipeline.predict(X_test)
y_prob_nb = nb_pipeline.predict_proba(X_test)[:, 1]

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_nb))
print(confusion_matrix(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.63
ROC-AUC: 0.6242857142857142
[[ 26  34]
 [ 40 100]]
              precision    recall  f1-score   support

         bad       0.39      0.43      0.41        60
        good       0.75      0.71      0.73       140

    accuracy                           0.63       200
   macro avg       0.57      0.57      0.57       200
weighted avg       0.64      0.63      0.63       200



In [7]:
dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=None,
        min_samples_split=10,
        class_weight="balanced",
        random_state=42
    ))
])

dt_pipeline.fit(X_train, y_train)

y_pred_dt = dt_pipeline.predict(X_test)
y_prob_dt = dt_pipeline.predict_proba(X_test)[:, 1]

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_dt))
print(confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Decision Tree Accuracy: 0.6
ROC-AUC: 0.589404761904762
[[28 32]
 [48 92]]
              precision    recall  f1-score   support

         bad       0.37      0.47      0.41        60
        good       0.74      0.66      0.70       140

    accuracy                           0.60       200
   macro avg       0.56      0.56      0.55       200
weighted avg       0.63      0.60      0.61       200



In [8]:
import joblib

joblib.dump(knn_pipeline, "../models/knn.pkl")
joblib.dump(nb_pipeline, "../models/naive_bayes.pkl")
joblib.dump(dt_pipeline, "../models/decision_tree.pkl")

['../models/decision_tree.pkl']

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

def get_metrics(model_name, y_true, y_pred, y_prob):
    report = classification_report(y_true, y_pred, output_dict=True)

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "Precision": report["weighted avg"]["precision"],
        "Recall": report["weighted avg"]["recall"],
        "F1_score": report["weighted avg"]["f1-score"]
    }

In [9]:
metrics_list = []

# KNN
y_pred_knn = knn_pipeline.predict(X_test)
y_prob_knn = knn_pipeline.predict_proba(X_test)[:,1]

metrics_list.append(
    get_metrics("KNN", y_test, y_pred_knn, y_prob_knn)
)

# Naive Bayes
y_pred_nb = nb_pipeline.predict(X_test)
y_prob_nb = nb_pipeline.predict_proba(X_test)[:,1]

metrics_list.append(
    get_metrics("Naive Bayes", y_test, y_pred_nb, y_prob_nb)
)

# Decision Tree
y_pred_dt = dt_pipeline.predict(X_test)
y_prob_dt = dt_pipeline.predict_proba(X_test)[:,1]

metrics_list.append(
    get_metrics("Decision Tree", y_test, y_pred_dt, y_prob_dt)
)

NameError: name 'get_metrics' is not defined

In [ ]:
import pandas as pd

metrics_df = pd.DataFrame(metrics_list)

metrics_df

In [ ]:
metrics_df.to_json("model_metrics.json", orient="records")